# Formula 1 2008 — Driver and Constructor Analysis

**Author:** Claudio Lombardi  
**Course:** Python Programming — ProfessionAI Master in Data Analytics

This project analyzes driver points, wins and podiums, generates driver and constructor standings, and exports driver standings to a text file.

The supplied educational dataset contains 180 records covering 10 drivers and 6 teams. Results refer only to those participants and do not represent the complete official championship standings.

Position 0 indicates a finish outside the points-paying top eight. Equal point totals retain the order of first appearance in the dataset; official championship tie-breaking rules are not implemented.

Run the cells in order in Google Colab. Enter a driver surname exactly as listed. Finish the interactive analysis by answering `no` or entering `quit` before proceeding to the standings.

# 1. Individual driver performance analysis

In [2]:
import pandas as pd
from google.colab import files

# Load the dataset once.
url = "https://proai-datasets.s3.eu-west-3.amazonaws.com/formula1_data.csv"
df = pd.read_csv(url)

# Points awarded under the assignment's 2008 scoring system.
points_system = {
    0: 0,
    1: 10,
    2: 8,
    3: 6,
    4: 5,
    5: 4,
    6: 3,
    7: 2,
    8: 1
}

# Calculate points once and reuse them in all analyses.
df["Points"] = df["Position"].map(points_system)

if df["Points"].isna().any():
    raise ValueError("The dataset contains an unexpected position.")


def analyze_driver(driver_name):
    """Return a list containing total points, wins and podiums."""
    driver_df = df[df["Driver"] == driver_name]

    if driver_df.empty:
        raise ValueError(f"Driver not found: {driver_name}")

    total_points = int(driver_df["Points"].sum())
    wins = int((driver_df["Position"] == 1).sum())
    podiums = int(driver_df["Position"].isin([1, 2, 3]).sum())

    return [total_points, wins, podiums]


def interactive_analysis():
    """Allow the user to select and analyze drivers repeatedly."""
    available_drivers = sorted(df["Driver"].unique())

    print("\nAvailable drivers:")
    for name in available_drivers:
        print("-", name)

    while True:
        driver = input(
            "\nEnter a driver surname (e.g. Hamilton), or quit: "
        ).strip()

        if driver.lower() == "quit":
            break

        if driver not in available_drivers:
            print("Driver not found. Use a surname exactly as listed.")
            continue

        points, wins, podiums = analyze_driver(driver)

        print(f"\nResults for {driver}:")
        print(f"Total points: {points}")
        print(f"Wins: {wins}")
        print(f"Podiums: {podiums}")

        again = input(
            "\nWould you like to analyze another driver? (yes/no): "
        ).strip().lower()

        if again != "yes":
            break

    print("\nDriver analysis completed.")


interactive_analysis()


Available drivers:
- Alonso
- Glock
- Hamilton
- Heidfeld
- Kovalainen
- Kubica
- Massa
- Raikkonen
- Trulli
- Vettel

Enter a driver surname (e.g. Hamilton), or quit: Alonso

Results for Alonso:
Total points: 61
Wins: 2
Podiums: 3

Would you like to analyze another driver? (yes/no): yes

Enter a driver surname (e.g. Hamilton), or quit: Massa

Results for Massa:
Total points: 97
Wins: 6
Podiums: 10

Would you like to analyze another driver? (yes/no): no

Driver analysis completed.


# 2. Driver standings and text export


In [3]:
def create_driver_standings():
    """Return driver standings and export them to a text file."""
    standings = {}

    for driver in df["Driver"].unique():
        standings[driver] = analyze_driver(driver)[0]

    # Sort by descending points.
    sorted_standings = dict(
        sorted(
            standings.items(),
            key=lambda item: item[1],
            reverse=True
        )
    )

    filename = "Drivers_Standings_2008.txt"

    with open(filename, "w", encoding="utf-8") as file:
        file.write("Drivers Standings 2008 Formula 1\n")

        for driver, points in sorted_standings.items():
            file.write(f"{driver}: {points}\n")

    print("\nDriver standings — supplied dataset:")
    for driver, points in sorted_standings.items():
        print(f"{driver}: {points}")

    print(f"\nStandings saved to '{filename}'.")

    return sorted_standings


driver_standings = create_driver_standings()
files.download("Drivers_Standings_2008.txt")


Driver standings — supplied dataset:
Hamilton: 98
Massa: 97
Raikkonen: 75
Kubica: 75
Alonso: 61
Heidfeld: 60
Kovalainen: 53
Vettel: 35
Trulli: 31
Glock: 25

Standings saved to 'Drivers_Standings_2008.txt'.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 3. Constructor standings

In [4]:
def create_constructor_standings():
    """Return constructor standings based on the supplied race records."""
    team_points = df.groupby("Team", sort=False)["Points"].sum()

    sorted_team_points = team_points.sort_values(
        ascending=False,
        kind="stable"
    )

    constructor_standings = {
        team: int(points)
        for team, points in sorted_team_points.items()
    }

    print("\nConstructor standings — supplied dataset:")
    for team, points in constructor_standings.items():
        print(f"{team}: {points}")

    return constructor_standings


constructor_standings = create_constructor_standings()


Constructor standings — supplied dataset:
Ferrari: 172
McLaren: 151
BMW: 135
Renault: 61
Toyota: 56
Toro Rosso: 35


In [5]:
# Verify that driver and constructor standings account for all points.
total_points = int(df["Points"].sum())

assert sum(driver_standings.values()) == total_points, (
    "Driver standings do not match the dataset total."
)

assert sum(constructor_standings.values()) == total_points, (
    "Constructor standings do not match the dataset total."
)

print(f"Validation passed: {total_points} points in both standings.")

Validation passed: 610 points in both standings.
